# Trajectory Generation

Source orientation: printed pages 325-352, PDF pages 343-370. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How do we turn geometric waypoints into a smooth, executable time history?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: path, trajectory, time scaling, cubic, quintic, via point, phase plane. The important conversions for this notebook are:

- A path names where to go; a trajectory names when to be there.
- Boundary conditions decide polynomial degree.
- Time-optimal scaling is a phase-plane feasibility problem.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. A column of joint angles may describe a point on a torus, a homogeneous transform may describe a rigid frame, and a matrix rank may reveal an instantaneous loss of motion. The notebook therefore pairs each formula with a small experiment: a plot, a residual, an ellipsoid, a path, a graph, or a constraint surface.

## Route Through the Notebook

1. Establish the objects and conventions needed for trajectory generation.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: time scaling and smooth paths.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-09-trajectory-generation/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| time scaling and smooth paths | time-scaling curves and phase limits | `artifacts/chapter-09-trajectory-generation/figures/trajectory-generation-lab.png` | how a path becomes a timed motion subject to boundary and speed limits |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-09-trajectory-generation/figures/trajectory-generation-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a residual, rank, eigenvalue, path length, or comparable margin that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For trajectory generation, the relevant failure modes are not side details; they are part of the concept. Singularities, chart boundaries, rank loss, time-scaling limits, contact mode changes, and wheel constraints are all examples of geometry asserting itself. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Compare cubic and quintic timing, then mark where acceleration constraints would bind.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, ask whether a rank should change, whether a path should lengthen, whether an eigenvalue should stay positive, whether a cone should widen, or whether a coordinate chart should approach a singular value. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- Smooth position alone is not enough for torque-controlled machines.
- Via points can create hidden velocity discontinuities.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, rank, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Trajectory generation separates geometry from timing.
- Boundary derivatives are design constraints, not afterthoughts.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 9,
  "slug": "chapter-09-trajectory-generation",
  "title": "Trajectory Generation",
  "notebook": "09-trajectory-generation.ipynb",
  "printed_start": 325,
  "printed_end": 352,
  "pdf_start": 343,
  "pdf_end": 370,
  "part_slug": "part-03-dynamics-trajectories-and-planning",
  "part_title": "Dynamics, Trajectories, and Planning",
  "theme": "planning",
  "visual_focus": "time scaling and smooth paths",
  "visual_kind": "time-scaling curves and phase limits",
  "artifact_stem": "trajectory-generation",
  "inspection_target": "how a path becomes a timed motion subject to boundary and speed limits",
  "question": "How do we turn geometric waypoints into a smooth, executable time history?",
  "terms": [
    "path",
    "trajectory",
    "time scaling",
    "cubic",
    "quintic",
    "via point",
    "phase plane"
  ],
  "translation": [
    "A path names where to go; a trajectory names when to be there.",
    "Boundary conditions decide polynomial degree.",
    "Time-optimal scaling is a phase-plane feasibility problem."
  ],
  "lab": "Compare cubic and quintic timing, then mark where acceleration constraints would bind.",
  "pitfalls": [
    "Smooth position alone is not enough for torque-controlled machines.",
    "Via points can create hidden velocity discontinuities."
  ],
  "takeaways": [
    "Trajectory generation separates geometry from timing.",
    "Boundary derivatives are design constraints, not afterthoughts."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Path, Trajectory, and Time Scaling

A useful way to keep the chapter straight is to separate three objects that often get blended together in casual speech. The path is the geometric trace: in joint space it might be a curve from one vector of joint angles to another, and in task space it might be a curve of rigid-body frames. The path parameter `s` is a coordinate along that trace, usually normalized so `s = 0` means the start and `s = 1` means the finish. The trajectory is what happens after a clock is attached: `q(t) = q(s(t))` or `X(t) = X(s(t))`. Once the clock appears, velocity and acceleration are no longer properties of the drawn path alone; they are products of the path shape and the time-scaling derivatives.

For a straight joint-space move, the chain rule makes the design choice visible. If `q(s) = q0 + s (q1 - q0)`, then `qdot(t) = sdot(t) (q1 - q0)` and `qddot(t) = sddot(t) (q1 - q0)`. A cubic time scaling can make the endpoint velocities zero, but it generally starts and ends with nonzero acceleration. A quintic time scaling has enough coefficients to also make endpoint accelerations zero. That extra smoothness matters when a downstream controller or actuator model treats acceleration as a demand rather than an afterthought. The geometric path did not change; the way the robot enters and leaves the path did.

The chapter's phase-plane viewpoint is the same idea with constraints drawn over it. Instead of asking only for a pretty polynomial, we ask which pairs `(s, sdot)` are dynamically feasible when actuator, velocity, and acceleration limits are present. A boundary curve marks where increasing speed would violate some constraint. A time-optimal policy usually alternates between maximum acceleration and maximum braking while staying inside the feasible region. The result is not just a scalar function; it is a certificate that the timing respects the machine.

Via points add one more caution. They are useful when a tool must pass near an inspection point or avoid an obstacle, but stitching polynomial segments together can hide derivative jumps. Position continuity alone gives a curve that looks connected while commanding an impossible change in velocity or acceleration at the knot. In a real trajectory generator, each via point should therefore specify which derivatives are continuous, which derivatives may reset, and whether the timing is being chosen globally or one segment at a time.


## Worked Example

The worked example below uses a straight joint-space path and two clocks. The path is identical in both cases, so any difference in velocity or acceleration comes from the time scaling alone. This is the core computational habit for the chapter: first decide what geometric curve should be followed, then inspect how the scalar clock changes endpoint derivatives and peak demands.


In [ ]:
from utils.planning import cubic_time_scaling, quintic_time_scaling

T = 2.0
t = np.linspace(0.0, T, 401)
tau = t / T
q0 = np.array([0.2, -0.6, 0.4])
q1 = np.array([1.1, 0.3, -0.2])
delta_q = q1 - q0

s_cubic = cubic_time_scaling(t, T)
s_quintic = quintic_time_scaling(t, T)
q_cubic = q0 + s_cubic[:, None] * delta_q
q_quintic = q0 + s_quintic[:, None] * delta_q

sdot_cubic = (6.0 * tau - 6.0 * tau**2) / T
sddot_cubic = (6.0 - 12.0 * tau) / T**2
sdot_quintic = (30.0 * tau**2 - 60.0 * tau**3 + 30.0 * tau**4) / T
sddot_quintic = (60.0 * tau - 180.0 * tau**2 + 120.0 * tau**3) / T**2

qdot_cubic = sdot_cubic[:, None] * delta_q
qddot_cubic = sddot_cubic[:, None] * delta_q
qdot_quintic = sdot_quintic[:, None] * delta_q
qddot_quintic = sddot_quintic[:, None] * delta_q

path_distance = np.linalg.norm(delta_q)
quintic_arc_length = np.trapz(np.linalg.norm(qdot_quintic, axis=1), t)
worked_example = {
    "path_distance": float(path_distance),
    "quintic_integrated_speed_error": float(abs(quintic_arc_length - path_distance)),
    "cubic_endpoint_acceleration_norm": float(max(np.linalg.norm(qddot_cubic[0]), np.linalg.norm(qddot_cubic[-1]))),
    "quintic_endpoint_acceleration_norm": float(max(np.linalg.norm(qddot_quintic[0]), np.linalg.norm(qddot_quintic[-1]))),
    "cubic_peak_speed": float(np.linalg.norm(qdot_cubic, axis=1).max()),
    "quintic_peak_speed": float(np.linalg.norm(qdot_quintic, axis=1).max()),
    "same_geometric_endpoint_error": float(np.linalg.norm(q_cubic[-1] - q_quintic[-1])),
}
assert worked_example["same_geometric_endpoint_error"] < 1e-12
assert worked_example["quintic_endpoint_acceleration_norm"] < 1e-12
assert worked_example["quintic_integrated_speed_error"] < 1e-4
worked_example


## Applied Lab

The applied lab stretches the same quintic move over several durations. Before running it, predict how the largest velocity and acceleration should scale. The path length stays fixed because the start and goal are unchanged, while the clock changes the demands: peak speed should scale like `1 / T`, and peak acceleration should scale like `1 / T^2`. This is why slowing a motion down is often the simplest way to recover feasibility when a phase-plane constraint is exceeded.


In [ ]:
durations = np.array([1.0, 2.0, 4.0])
profiles = []
for duration in durations:
    grid = np.linspace(0.0, duration, 401)
    tau_grid = grid / duration
    sdot = (30.0 * tau_grid**2 - 60.0 * tau_grid**3 + 30.0 * tau_grid**4) / duration
    sddot = (60.0 * tau_grid - 180.0 * tau_grid**2 + 120.0 * tau_grid**3) / duration**2
    qdot = sdot[:, None] * delta_q
    qddot = sddot[:, None] * delta_q
    profiles.append({
        "duration": float(duration),
        "peak_speed": float(np.linalg.norm(qdot, axis=1).max()),
        "peak_acceleration": float(np.linalg.norm(qddot, axis=1).max()),
    })

lab_summary = {
    "theme": CHAPTER["theme"],
    "profiles": profiles,
    "speed_ratio_T1_to_T4": float(profiles[0]["peak_speed"] / profiles[-1]["peak_speed"]),
    "acceleration_ratio_T1_to_T4": float(profiles[0]["peak_acceleration"] / profiles[-1]["peak_acceleration"]),
}
assert abs(lab_summary["speed_ratio_T1_to_T4"] - 4.0) < 1e-9
assert abs(lab_summary["acceleration_ratio_T1_to_T4"] - 16.0) < 1e-9
lab_summary


## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
assert worked_example['quintic_endpoint_acceleration_norm'] < 1e-12
assert worked_example['same_geometric_endpoint_error'] < 1e-12
assert abs(lab_summary['speed_ratio_T1_to_T4'] - 4.0) < 1e-9
assert abs(lab_summary['acceleration_ratio_T1_to_T4'] - 16.0) < 1e-9
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity
